# 03 Data Validation

> Change only `RAW` / `PROCESSED` paths if your folder location is different.

In [6]:
from pathlib import Path
import pandas as pd
import numpy as np
import re, json

BASE = Path(r"C:\CHANGE\THIS\TO\YOUR\PROJECT")
RAW = BASE / "data" / "raw"
PROCESSED = BASE / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)
arrivals = pd.read_csv(RAW / "C:\\Users\\yalamanchi rohitha\\Downloads\\track3_agritech_dataset_files\\track3_mandi_arrivals.csv")
master = pd.read_csv(RAW / "C:\\Users\\yalamanchi rohitha\\Downloads\\track3_agritech_dataset_files\\track3_mandi_master.csv")
transport = pd.read_csv(RAW / "C:\\Users\\yalamanchi rohitha\\Downloads\\track3_agritech_dataset_files\\track3_transport_logistics.csv")
master
with open(RAW / "C:\\Users\\yalamanchi rohitha\\Downloads\\track3_agritech_dataset_files\\track3_price_and_msp.json", encoding="utf-8") as f:
    prices = pd.DataFrame(json.load(f))

weather = pd.read_excel(RAW / "C:\\Users\yalamanchi rohitha\\Downloads\\track3_agritech_dataset_files\\track3_weather_sensors.xlsx")

## Validation checks

In [7]:
print("\nNumeric sanity:")

# Convert numeric columns safely before checking
if "arrival_quantity_qtl" in arrivals.columns:
    arrivals["arrival_quantity_qtl"] = pd.to_numeric(
        arrivals["arrival_quantity_qtl"],
        errors="coerce"
    )
    print(
        "Negative arrival qtl:",
        (arrivals["arrival_quantity_qtl"] < 0).sum()
    )
else:
    print("Negative arrival qtl: Column not found")


if "modal_price" in prices.columns:
    prices["modal_price"] = (
        prices["modal_price"]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("₹", "", regex=False)
        .str.replace("Rs.", "", regex=False)
        .str.replace("INR", "", regex=False)
        .str.strip()
    )

    prices["modal_price"] = pd.to_numeric(
        prices["modal_price"],
        errors="coerce"
    )

    print(
        "Negative price:",
        (prices["modal_price"] < 0).sum()
    )
else:
    print("Negative price: Column not found")


if "distance_km" in transport.columns:
    transport["distance_km"] = pd.to_numeric(
        transport["distance_km"],
        errors="coerce"
    )
    print(
        "Negative distance:",
        (transport["distance_km"] < 0).sum()
    )
else:
    print("Negative distance: Column not found")


if "transit_hours" in transport.columns:
    transport["transit_hours"] = pd.to_numeric(
        transport["transit_hours"],
        errors="coerce"
    )
    print(
        "Negative transit:",
        (transport["transit_hours"] < 0).sum()
    )
else:
    print("Negative transit: Column not found")


Numeric sanity:
Negative arrival qtl: Column not found
Negative price: 0
Negative distance: Column not found
Negative transit: 563


## Final validation status

In [9]:
# ==========================================
# FINAL VALIDATION CHECKS
# ==========================================

checks = {}

# Duplicate checks
checks["arrivals_no_duplicates"] = arrivals.duplicated().sum() == 0
checks["prices_no_duplicates"] = prices.duplicated().sum() == 0
checks["weather_no_duplicates"] = weather.duplicated().sum() == 0
checks["transport_no_duplicates"] = transport.duplicated().sum() == 0
checks["master_no_duplicates"] = master.duplicated().sum() == 0


# Arrival quantity check
if "arrival_quantity_qtl" in arrivals.columns:
    quantity = pd.to_numeric(
        arrivals["arrival_quantity_qtl"],
        errors="coerce"
    )
    checks["no_negative_arrivals"] = (quantity < 0).sum() == 0
else:
    checks["arrival_quantity_qtl_exists"] = False


# Price check
if "modal_price" in prices.columns:
    price = pd.to_numeric(
        prices["modal_price"],
        errors="coerce"
    )
    checks["no_negative_prices"] = (price < 0).sum() == 0
else:
    checks["modal_price_exists"] = False


# Distance check
if "distance_km" in transport.columns:
    distance = pd.to_numeric(
        transport["distance_km"],
        errors="coerce"
    )
    checks["no_negative_distance"] = (distance < 0).sum() == 0
else:
    checks["distance_km_exists"] = False


# Transit check
if "transit_hours" in transport.columns:
    transit = pd.to_numeric(
        transport["transit_hours"],
        errors="coerce"
    )
    checks["no_negative_transit"] = (transit < 0).sum() == 0
else:
    checks["transit_hours_exists"] = False


# Display validation results
validation_results = pd.DataFrame(
    list(checks.items()),
    columns=["check", "passed"]
)

display(validation_results)

,check,passed
0,arrivals_no_duplicates,False
1,prices_no_duplicates,True
2,weather_no_duplicates,True
3,transport_no_duplicates,False
4,master_no_duplicates,False
5,arrival_quantity_qtl_exists,False
6,no_negative_prices,True
7,distance_km_exists,False
8,no_negative_transit,False
